# Agents as subagents

In the last notebook, tools were Python functions. This notebook makes one move:

> **An agent is a callable that takes text and returns text. So an agent can be a tool.**

Wrap agent B in `@tool` and hand it to agent A, and A can now delegate. A becomes a
**supervisor**; B becomes a **subagent**.

**What we build:** a content studio. One editor-in-chief coordinating a researcher, a writer and an
SEO reviewer — the same job as notebook 3's writer agent, but split across four specialists.

Prerequisite: notebook `03_LangChain_Agents_Complete`. Same versions: `langchain 1.3`, `langgraph 1.2`.

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))          # OPENAI_API_KEY, TAVILY_API_KEY

import os, time, json
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)
print("ready")

ready


---
## 1. Why not just one agent with all the tools?

That is the right first question, and the honest answer is: for five tools, one agent is better.
Do not reach for subagents because they look sophisticated.

Here is the version that breaks. Give one agent every tool the studio needs:

In [2]:
from tavily import TavilyClient
tavily = TavilyClient(os.environ["TAVILY_API_KEY"])


@tool
def web_search(query: str) -> str:
    """Search the web for current facts, news and figures. Returns titles, snippets and URLs."""
    hits = tavily.search(query, max_results=5)["results"]
    # Realistic: a real search tool hands back everything it got, not a tidy summary.
    return "\n\n".join(f"{h['title']}\n{h['content']}\n{h['url']}" for h in hits)


@tool
def word_count(text: str) -> int:
    """Count the number of words in a piece of text."""
    return len(text.split())


@tool
def keyword_density(text: str, keyword: str) -> str:
    """Report how often a keyword appears in a text, as a percentage of total words."""
    words = text.lower().split()
    n = sum(1 for w in words if keyword.lower() in w)
    return f"'{keyword}' appears {n} times in {len(words)} words ({n / max(len(words), 1):.1%})"


@tool
def readability(text: str) -> str:
    """Estimate reading difficulty from average sentence length."""
    sentences = [s for s in text.replace("!", ".").replace("?", ".").split(".") if s.strip()]
    avg = len(text.split()) / max(len(sentences), 1)
    verdict = "easy" if avg < 16 else "dense" if avg > 24 else "moderate"
    return f"{avg:.1f} words per sentence - {verdict}"


BANNED = ["revolutionary", "game-changing", "unleash", "seamless", "leverage", "synergy"]


@tool
def check_banned_words(draft: str) -> str:
    """Check a draft against the house style banned-word list."""
    hits = [w for w in BANNED if w in draft.lower()]
    return "clean" if not hits else f"REWRITE, banned words present: {hits}"


all_tools = [web_search, word_count, keyword_density, readability, check_banned_words]
print("one agent, all tools:", [t.name for t in all_tools])

one agent, all tools: ['web_search', 'word_count', 'keyword_density', 'readability', 'check_banned_words']


In [3]:
flat = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=(
        "You are a content studio. Research the topic with web_search, write a short post, "
        "then check it with check_banned_words, readability, keyword_density and word_count. "
        "Fix anything the checks complain about. Return the final post."
    ),
)

BRIEF = "Write a short post on what a vector database is, for developers. Target keyword: embeddings."

t0 = time.time()
flat_out = flat.invoke({"messages": [{"role": "user", "content": BRIEF}]})
flat_secs = time.time() - t0

flat_chars = sum(len(str(m.content)) for m in flat_out["messages"])
print(f"messages in the single context : {len(flat_out['messages'])}")
print(f"characters in that context     : {flat_chars:,}")
print(f"seconds                        : {flat_secs:.1f}")

messages in the single context : 9
characters in that context     : 6,585
seconds                        : 17.3


It works. Look at what it cost, though: every search result, every check, every rewrite is now
sitting in **one** message list. And that list is re-sent to the model on every single turn of the
loop (notebook 3, section 4).

Three specific failures show up as you add tools:

1. **Context bloat.** Raw search results are long and are only needed *once* — to write the draft.
   They stay in context for the rest of the run, and you pay for them on every subsequent turn.
2. **Tool confusion.** Five tools all compete in one prompt. Ten or fifteen, and the model starts
   picking the wrong one, or forgetting one exists.
3. **One prompt, many jobs.** "Research thoroughly, cite sources" and "write tightly, no hedging"
   are opposite instructions. Cramming both into one `system_prompt` means each dilutes the other.

Subagents fix all three by giving each job its own context window, its own tools and its own prompt.

---
## 2. A subagent is just an agent

Nothing new here — this is `create_agent` from notebook 3. What makes it a *sub*agent is only how
we call it, in section 3.

Note how sharp the prompt can be now that this agent has exactly one job:

In [4]:
researcher = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "You are a research assistant. Search the web, then return 4-6 bullet points of "
        "concrete facts. Every bullet must end with its source URL in brackets. "
        "No preamble, no conclusion, no opinions. If you cannot verify a fact, leave it out."
    ),
)

r = researcher.invoke({"messages": [{"role": "user", "content": "What is a vector database?"}]})
print(r["messages"][-1].content)
print(f"\n[researcher used {len(r['messages'])} messages internally]")

- A vector database stores, manages, and indexes high-dimensional vector data, where data points are stored as arrays of numbers called "vectors" that can be compared and clustered based on similarity, enabling low-latency queries ideal for AI applications. [https://www.databricks.com/blog/what-is-vector-database]

- Vector databases are designed to handle complex data types such as images, videos, or other multidimensional data, providing flexibility for AI and machine learning use cases. [https://www.databricks.com/blog/what-is-vector-database]

- Vector databases enable similarity search by determining which data vectors are near a given input query vector, supporting use cases like recommendation systems, image and video search, and natural language processing. [https://developers.cloudflare.com/vectorize/reference/what-is-a-vector-database]

- Some vector databases separate the storage of data objects from vector indexes, allowing multiple indexes to point to the same object, comb

---
## 3. Wrap it with `@tool`

This is the whole pattern. Three lines:

In [5]:
@tool("research_topic", description=(
    "Research a topic on the web and return bullet-point facts with source URLs. "
    "Use this before writing anything that needs facts, figures or current events."
))
def research_topic(topic: str) -> str:
    result = researcher.invoke({"messages": [{"role": "user", "content": topic}]})
    return result["messages"][-1].content


print("name       :", research_topic.name)
print("args       :", research_topic.args)
print("description:", research_topic.description)

name       : research_topic
args       : {'topic': {'title': 'Topic', 'type': 'string'}}
description: Research a topic on the web and return bullet-point facts with source URLs. Use this before writing anything that needs facts, figures or current events.


Read that function body again, because it is the entire idea:

```python
result = researcher.invoke({"messages": [...]})   # run the subagent
return result["messages"][-1].content             # return ONLY its final answer
```

The supervisor sees a tool that takes a string and returns a string. It has no idea an entire agent
loop — with its own model calls, its own tool calls, its own dead ends — ran inside.

**Why `@tool("research_topic", description=...)` and not a docstring?** Both work. But this
description is not documentation, it is the **routing instruction** the supervisor reads to decide
whether to delegate. Writing it as an explicit argument, next to the name, keeps it where you will
maintain it. Note it says *when* to use the subagent, not what the subagent is.

Now the other two specialists, wrapped the same way.

In [6]:
writer_agent = create_agent(
    model=llm,
    tools=[check_banned_words],
    system_prompt=(
        "You are a staff writer for a developer audience. Plain, technical, short sentences, "
        "no hype, second person avoided. Write only from the facts you are given - never add "
        "figures of your own. Always run check_banned_words on your draft and fix any hits. "
        "Return the post body only: no title, no commentary."
    ),
)

seo_agent = create_agent(
    model=llm,
    tools=[word_count, keyword_density, readability],
    system_prompt=(
        "You are an SEO reviewer. Run all three of your tools on the draft, then return: "
        "the numbers you measured, a verdict of PASS or FIX, and if FIX, at most three "
        "specific edits. Do not rewrite the post yourself."
    ),
)


@tool("write_section", description=(
    "Write a post from research notes. Pass the topic, the notes to use, and a word budget. "
    "Returns the post body. Use this after research_topic."
))
def write_section(topic: str, notes: str, word_budget: int) -> str:
    result = writer_agent.invoke({"messages": [{"role": "user", "content":
        f"Topic: {topic}\nWord budget: about {word_budget} words.\n\nFacts to use:\n{notes}"}]})
    return result["messages"][-1].content


@tool("seo_review", description=(
    "Review a draft for length, keyword density and readability. Returns measurements and a "
    "PASS/FIX verdict with suggested edits. Use this on every draft before finishing."
))
def seo_review(draft: str, target_keyword: str) -> str:
    result = seo_agent.invoke({"messages": [{"role": "user", "content":
        f"Target keyword: {target_keyword}\n\nDraft:\n{draft}"}]})
    return result["messages"][-1].content


subagent_tools = [research_topic, write_section, seo_review]
for t in subagent_tools:
    print(f"{t.name:16s} {list(t.args.keys())}")

research_topic   ['topic']
write_section    ['topic', 'notes', 'word_budget']
seo_review       ['draft', 'target_keyword']


Look at the argument lists. `research_topic` takes a topic. `write_section` takes a topic, notes and
a budget. Those are not accidents — **a subagent's tool signature is its contract**, and it is where
you decide how much context flows down:

| Signature | Effect |
|---|---|
| `(topic: str)` | cheapest, most isolated. The subagent knows nothing else. |
| `(topic, notes, word_budget)` | supervisor must pass what it learned. Explicit, controllable. |
| pass the whole message history | subagent sees everything — and you have thrown away the isolation you came for. |

Prefer the narrowest signature that does the job. If a subagent needs three arguments to work, three
arguments is the right answer — but every extra one is context you are copying, and paying for twice.

---
## 4. The supervisor

Now the top-level agent. Its tools are the three subagents, and its `system_prompt` is no longer
about writing at all — it is about **delegation order**. That shift is the payoff.

In [7]:
class Post(BaseModel):
    """A finished post, ready for the CMS."""
    title: str = Field(description="Headline, under 12 words.")
    body: str = Field(description="The post body, from write_section.")
    sources: list[str] = Field(description="Source URLs from research_topic.")
    seo_verdict: str = Field(description="The final PASS/FIX verdict from seo_review.")


EDITOR_SYSTEM = """You are the editor-in-chief of a content studio. You do not research, write
or measure anything yourself - you delegate, then judge the results.

For every brief:
1. research_topic - get the facts.
2. write_section - pass the topic, the research notes, and the word budget.
3. seo_review - review the draft against the target keyword.
4. If the verdict is FIX, call write_section again with the reviewer's edits added to the notes.
   Do this at most twice, then ship the best draft you have.

Return the Post. Sources must come from the research notes - never invent a URL."""

editor = create_agent(
    model=llm,
    tools=subagent_tools,
    system_prompt=EDITOR_SYSTEM,
    response_format=Post,
    checkpointer=InMemorySaver(),
)

thread = {"configurable": {"thread_id": "studio-vectordb"}}
print("supervisor tools:", [t.name for t in subagent_tools])

supervisor tools: ['research_topic', 'write_section', 'seo_review']


In [8]:
t0 = time.time()
out = editor.invoke({"messages": [{"role": "user", "content":
    "Write a ~130 word post on what a vector database is, for developers. "
    "Target keyword: embeddings."}]}, thread)
sub_secs = time.time() - t0

post = out["structured_response"]
print(f"TITLE   : {post.title}")
print(f"SEO     : {post.seo_verdict}")
print(f"SOURCES : {len(post.sources)}")
for s in post.sources:
    print(f"          {s}")
print(f"\n{post.body}")

TITLE   : What Is a Vector Database? A Developer's Guide to Embeddings
SEO     : PASS
SOURCES : 5
          https://cloud.google.com/discover/what-is-a-vector-database
          https://www.databricks.com/blog/what-is-vector-database
          https://learn.microsoft.com/en-us/dotnet/ai/vector-stores/overview
          https://www.deeplearning.ai/courses/vector-databases-embeddings-applications
          https://medium.com/@vladris/embeddings-and-vector-databases-732f9927b377

Vector databases store, index, and query vector embeddings, which are numerical representations of unstructured data such as text, images, or audio. These embeddings are arrays of floating-point numbers that convert complex data into a format suitable for machine learning models. Vector databases enable efficient similarity search and support semantic queries using embeddings. They provide CRUD operations, metadata filtering, horizontal scaling, and maintain data integrity and security. These features make vector

### 4a. Watch the delegation happen

`stream_mode="updates"` shows the supervisor's decisions. Everything the subagents did internally is
invisible here — which is exactly the point.

In [9]:
for chunk in editor.stream({"messages": [{"role": "user", "content":
        "Now a ~90 word post on what an embedding model is. Target keyword: embeddings."}]},
        {"configurable": {"thread_id": "studio-embeddings"}}, stream_mode="updates"):
    for node, update in chunk.items():
        msg = update["messages"][-1]
        calls = [c["name"] for c in getattr(msg, "tool_calls", [])]
        if calls:
            print(f"[{node:6s}] delegates -> {calls}")
        else:
            print(f"[{node:6s}] {type(msg).__name__}: {str(msg.content)[:70]}")

[model ] delegates -> ['research_topic']


[tools ] ToolMessage: - Embedding models convert data such as text, images, or user behavior


[model ] delegates -> ['write_section']


[tools ] ToolMessage: Embedding models convert data like text, images, or user behavior into


[model ] delegates -> ['seo_review']


[tools ] ToolMessage: Word count: 83
Keyword density for "embeddings": 3.6%
Readability (ave


[model ] AIMessage: {"title":"What Are Embeddings in Embedding Models?","body":"Embedding 


---
## 5. Measuring the claim

Section 1 claimed subagents keep the supervisor's context small. Do not take that on faith — measure
it. First, one brief through each design:

In [10]:
sub_chars = sum(len(str(m.content)) for m in out["messages"])

print(f"{'':22s} {'messages':>9s} {'characters':>12s} {'seconds':>9s}")
print(f"{'flat agent (sec 1)':22s} {len(flat_out['messages']):>9d} {flat_chars:>12,d} {flat_secs:>9.1f}")
print(f"{'supervisor (sec 4)':22s} {len(out['messages']):>9d} {sub_chars:>12,d} {sub_secs:>9.1f}")

# Where did the raw research go?
notes = [m for m in out["messages"]
         if type(m).__name__ == "ToolMessage" and m.name == "research_topic"][0]
raw = researcher.invoke({"messages": [{"role": "user", "content": "What is a vector database?"}]})
raw_chars = sum(len(str(m.content)) for m in raw["messages"])

print(f"\nresearcher's own context, internally      : {raw_chars:,} characters")
print(f"what it handed up to the supervisor       : {len(str(notes.content)):,} characters")
print(f"discarded on return                       : {raw_chars - len(str(notes.content)):,} characters")

                        messages   characters   seconds
flat agent (sec 1)             9        6,585      17.3
supervisor (sec 4)             8        4,466      33.5



researcher's own context, internally      : 6,863 characters
what it handed up to the supervisor       : 1,705 characters
discarded on return                       : 5,158 characters


The middle line is the pattern: the researcher burned **6.9k characters** internally — five full
search results, its own reasoning, a retry — and handed up **1.7k** of bullet points. Roughly
**three quarters of its context was thrown away on return**, and never entered the supervisor's
message list. That is what "context isolation" means, concretely.

So even on one brief the supervisor's context is the smaller of the two. But look at the seconds
column before you celebrate: it took about **twice as long**. Four agents means four sets of model
calls, and the notes travel down to the writer and back up again.

That is the real trade, and it cuts both ways:

- **You gain** a context that holds *conclusions* rather than raw material.
- **You lose** the details. The supervisor cannot re-read a search snippet it never saw. When it
  needs to, that is a signal the subagent should be returning more.
- **You pay in latency and total tokens.** On a single short brief, the flat agent is the better
  engineering choice. Be honest about that.

(Your numbers will differ — live search results and a non-zero-temperature loop change every run.
The *shape* is what reproduces, not the digits.)

### 5a. Where the win actually is

One brief is the wrong test, because a context problem is a problem that *accumulates*. So run a
three-turn conversation through both designs and watch the message list grow.

Both agents get a checkpointer and a thread, so this is the same setup a real chat product has:

In [11]:
flat_mem = create_agent(
    model=llm, tools=all_tools,
    system_prompt=(
        "You are a content studio. Research the topic with web_search, write a short post, "
        "then check it with check_banned_words, readability, keyword_density and word_count. "
        "Fix anything the checks complain about. Return the final post."
    ),
    checkpointer=InMemorySaver(),
)

TURNS = [
    "Write a ~90 word post on what a vector database is. Target keyword: embeddings.",
    "Now one on what an embedding model is. Target keyword: embeddings.",
    "Now one on when NOT to use a vector database. Target keyword: embeddings.",
]


def context_size(agent, cfg):
    msgs = agent.get_state(cfg).values["messages"]
    return len(msgs), sum(len(str(m.content)) for m in msgs)


flat_cfg = {"configurable": {"thread_id": "growth-flat"}}
sup_cfg = {"configurable": {"thread_id": "growth-sup"}}

print(f"{'turn':>4s} | {'flat msgs':>9s} {'flat chars':>11s} | {'sup msgs':>8s} {'sup chars':>10s}")
print("-" * 56)
for i, t in enumerate(TURNS, 1):
    flat_mem.invoke({"messages": [{"role": "user", "content": t}]}, flat_cfg)
    editor.invoke({"messages": [{"role": "user", "content": t}]}, sup_cfg)
    fm, fc = context_size(flat_mem, flat_cfg)
    sm, sc = context_size(editor, sup_cfg)
    print(f"{i:>4d} | {fm:>9d} {fc:>11,d} | {sm:>8d} {sc:>10,d}")

turn | flat msgs  flat chars | sup msgs  sup chars
--------------------------------------------------------


Deserializing unregistered type __main__.Post from checkpoint. This will be blocked in a future version. Set LANGGRAPH_STRICT_MSGPACK=true to block now, or add to allowed_msgpack_modules to allow explicitly: [('__main__', 'Post')]


   1 |         9       5,625 |        8      3,867


   2 |        21      11,857 |       16      7,601


   3 |        33      18,114 |       24     10,622


There it is. Both grow — a thread only grows — but they grow at **different rates**, and the gap is
what matters:

| after turn | flat context | supervisor context | gap |
|---|---|---|---|
| 1 | ~5.6k chars | ~3.9k chars | 1.8k |
| 2 | ~11.9k chars | ~7.6k chars | 4.3k |
| 3 | ~18.1k chars | ~10.6k chars | 7.5k |

The flat agent adds roughly **6k characters per turn**, the supervisor about **3.5k** — because the
flat agent's raw search results stay in the thread forever, while the supervisor only ever kept the
bullet points. The gap nearly doubled with each turn, and every character in it is re-sent on
**every model call of every future turn**.

That is the answer to "when do subagents pay off":

- **Turn 1:** the flat agent wins. Simpler, faster, cheaper.
- **Turn 3:** roughly even, once you price the re-sent context.
- **Turn 10+:** the flat agent is dragging ten sets of search results through every call. This is
  where the pattern earns its keep — and it is why the trade is about *conversations*, not *calls*.

So the decision rule is not "is my task complex?" It is: **how much raw material does this work
generate that nobody needs afterwards?** A lot → subagents. Very little → one agent.

---
## 6. Memory lives in the supervisor only

The supervisor has a checkpointer. The subagents do not. So a follow-up on the same thread works —
and it works *without* the subagents remembering anything:

> The *"Deserializing unregistered type `__main__.Post`"* warning is harmless and expected: the
> checkpointer is saving your Pydantic object, and a class defined in a notebook cell isn't
> importable by name. In a real project `Post` lives in a module and the warning goes away.

In [12]:
follow = editor.invoke({"messages": [{"role": "user", "content":
    "Good. Now cut it to 70 words and make the title more direct. Same sources."}]}, thread)

p2 = follow["structured_response"]
print(f"TITLE : {p2.title}")
print(f"WORDS : {len(p2.body.split())}  (was {len(post.body.split())})")
print(f"\n{p2.body}")

TITLE : Vector Databases Explained: Managing Embeddings for AI
WORDS : 72  (was 126)

Vector databases store, index, and query vector embeddings, numerical representations of unstructured data such as text, images, or audio. These embeddings convert complex data into formats suitable for machine learning models. Vector databases enable similarity search and semantic queries using embeddings. They support CRUD operations, metadata filtering, and scaling. Essential for AI, they convert prompts into embeddings, run vector searches, and assist language models. Applications include NLP, image recognition, and recommender systems.


In [13]:
# Proof the subagent itself is stateless: ask it directly what we just discussed.
amnesia = researcher.invoke({"messages": [{"role": "user", "content":
    "What topic did I just ask you to research?"}]})
print("researcher:", amnesia["messages"][-1].content[:200])

print("\nsupervisor thread messages:", len(editor.get_state(thread).values["messages"]))

researcher: You asked me to research the topic of "What topic did I just ask you to research?" which is a meta-question about your previous query. However, you have not asked me to research any specific topic yet

supervisor thread messages: 14


That split is deliberate and it is the safest default:

- **Supervisor: stateful.** It owns the conversation, the `thread_id`, and the user relationship.
- **Subagents: stateless.** Each invocation is a clean room. Same input, same behaviour, every time.

Which means anything a subagent needs to know, the supervisor must **pass down** — that is what
`write_section(topic, notes, word_budget)` is for. It feels redundant the first time you write it.
It is what makes subagents testable in isolation and safe to run in parallel.

Giving a subagent its own checkpointer is possible, but think hard first: two agents with two
memories of the same conversation will drift, and reconciling them is your problem.

---
## 7. What goes wrong

| Symptom | Cause | Fix |
|---|---|---|
| Supervisor answers itself instead of delegating | its prompt lets it | "You do not research or write yourself" — and give it no other tools |
| A subagent is never called | its tool `description` says *what it is*, not *when to use it* | rewrite as a trigger: "Use this before writing anything that needs facts" |
| Subagent gets a vague task and guesses | signature too narrow | add arguments — `notes`, `word_budget`, `tone` |
| Costs jumped 4x | you delegated work that fit in one context | go back to one agent; subagents earn their cost only when they discard a lot of context |
| Runs forever, re-reviewing | no loop bound | put a limit in the prompt ("at most twice"), and set `middleware` / recursion limits for a hard stop |
| Supervisor invents sources | it never saw them, so it filled the gap | make the subagent return them and say "never invent a URL" (both are in `EDITOR_SYSTEM`) |

**Depth:** stop at two levels. A subagent that delegates to a sub-subagent is nearly impossible to
debug, because each layer discards the context of the one below it.

**When *not* to use this pattern:**

- Fewer than ~6 tools, one coherent job → one agent.
- The steps are fixed and always the same → that is a chain or a graph, not a supervisor. You are
  paying an LLM to make a decision you already know the answer to.
- Two agents that need to converse with each other or with the user → look at **handoffs** instead
  of subagents.

---
## Recap

```python
sub = create_agent(model=llm, tools=[...], system_prompt="one sharp job")

@tool("sub_name", description="WHEN to use this, not what it is")
def call_sub(task: str, context: str) -> str:
    return sub.invoke({"messages": [{"role": "user", "content": f"{task}\n{context}"}]})["messages"][-1].content

supervisor = create_agent(
    model=llm,
    tools=[call_sub, ...],
    system_prompt="You delegate. Order: 1... 2... 3...",
    checkpointer=InMemorySaver(),   # supervisor remembers; subagents do not
)
```

Five things to remember:

1. A subagent is a normal agent. `@tool` is the only new code.
2. Return `result["messages"][-1].content` — the conclusion, not the transcript. That discard *is*
   the context isolation.
3. The tool `description` is a routing instruction. Write it as a trigger condition.
4. The tool signature is the contract. Narrowest that works; add arguments when the subagent guesses.
5. Memory belongs to the supervisor. Subagents are stateless, and that is a feature.

### Exercises

1. Change `research_topic`'s description to "Researches things." Rerun the editor. Count how often
   it skips research.
2. Drop `word_budget` from `write_section`. How far off the target length does the writer drift?
3. Add a `fact_check` subagent that re-searches every claim in the draft and returns
   `SUPPORTED`/`UNSUPPORTED` per claim. Wire it in after `seo_review`.
4. Run the flat agent and the supervisor on the same brief with LangSmith tracing on, and compare
   total tokens. At what brief size does the supervisor become cheaper?
5. Give `researcher` its own checkpointer and a fixed `thread_id`. Run three different briefs
   through the editor and explain the output you get.